In [ ]:
import sys, os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diploma'
    sys.path.insert(0, DRIVE_ROOT)

    get_ipython().system(f'pip install -q -r {DRIVE_ROOT}/requirements_aitoolkit_flux.txt')

    SCRIPT = '/content/train_dreambooth_lora_flux.py'
    SCRIPT_URL = ('https://raw.githubusercontent.com/huggingface/diffusers/main/'
                  'examples/dreambooth/train_dreambooth_lora_flux.py')
    get_ipython().system(f'wget -q -O {SCRIPT} {SCRIPT_URL}')
    get_ipython().system(f"sed -i 's/^check_min_version/# check_min_version/' {SCRIPT}")
    print(f'Training script: {SCRIPT}')

    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = os.environ.get('HF_TOKEN')

    from huggingface_hub import login
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        os.environ['HF_TOKEN'] = hf_token
        print('HF login: from Colab Secrets / env')
    else:
        print('HF_TOKEN not set in Colab Secrets - opening interactive login.')
        print('(Add a Secret named HF_TOKEN to skip this in future runtimes.)')
        login()
except ModuleNotFoundError:
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    sys.path.insert(0, DRIVE_ROOT)
    print('Local run - install deps in your venv first.')

print(f'\nDRIVE_ROOT: {DRIVE_ROOT}')

In [ ]:
import os, sys, runpy, json, shutil
from pathlib import Path
from tqdm.auto import tqdm

IS_COLAB = DRIVE_ROOT.startswith('/content/drive')

# FLUX uses a separate folder with PNG only (no .txt sidecar files).
# HF's training script tries to PIL.open every file in instance_data_dir
# and dies on .txt captions — so don't include them.
FLUX_TRAIN_DIR = '/content/data/flux_train_images' if IS_COLAB else f'{DRIVE_ROOT}/data/flux_train_images'

if not Path(FLUX_TRAIN_DIR).exists() or not list(Path(FLUX_TRAIN_DIR).glob('*.png')):
    Path(FLUX_TRAIN_DIR).mkdir(parents=True, exist_ok=True)
    with open(f'{DRIVE_ROOT}/data/train.jsonl', encoding='utf-8') as f:
        pairs = [json.loads(l) for l in f]
    for item in tqdm(pairs, desc='Copy images for FLUX'):
        stem = Path(item['png_path']).stem
        shutil.copy(item['png_path'], f'{FLUX_TRAIN_DIR}/{stem}.png')
    print(f'Prepared {len(pairs)} images -> {FLUX_TRAIN_DIR}')

OUTPUT_DIR = f'{DRIVE_ROOT}/results/experiments/flux_r16'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

SCRIPT = '/content/train_dreambooth_lora_flux.py'
if not Path(SCRIPT).exists():
    raise RuntimeError(f'{SCRIPT} missing - run cell-1-setup first.')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# DreamBooth-style training: one shared instance_prompt for the whole set
# (FLUX learns 'LOGOIMG' as the trigger token for "logo"). Per-image captions
# would require a HuggingFace Dataset, not a plain folder.
ARGV = [
    SCRIPT,
    '--pretrained_model_name_or_path=black-forest-labs/FLUX.1-dev',
    f'--instance_data_dir={FLUX_TRAIN_DIR}',
    f'--output_dir={OUTPUT_DIR}',
    '--instance_prompt=a LOGOIMG logo, vector, flat design, white background',
    '--mixed_precision=bf16',
    '--resolution=512',
    '--train_batch_size=1',
    '--gradient_accumulation_steps=4',
    '--gradient_checkpointing',
    '--optimizer=adamw',
    '--learning_rate=1e-4',
    '--lr_scheduler=cosine',
    '--lr_warmup_steps=100',
    '--max_train_steps=4000',
    '--checkpointing_steps=500',
    '--rank=16',
    '--seed=42',
    '--cache_latents',
]

print('Launching FLUX.1-dev LoRA training (rank=16, 4000 steps)...')
print('Args:', ' '.join(ARGV[1:]))

_old_argv = sys.argv
try:
    sys.argv = ARGV
    runpy.run_path(SCRIPT, run_name='__main__')
finally:
    sys.argv = _old_argv

print('\nDONE')

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from pathlib import Path

TEST_PROMPTS = [
    'minimalist coffee shop logo, circular icon, dark green, flat vector design, white background',
    'tech startup logo, geometric hexagon, blue gradient, bold sans-serif, white background',
    'bakery logo, wheat sheaf icon, warm brown, handcrafted artisan style, white background',
    'fitness brand, lion silhouette, orange and black, bold geometric, white background',
    'law firm logo, balanced scales, navy blue, serif elegant typography, white background',
]

out_dir = f'{DRIVE_ROOT}/results/experiments/sd15_baseline'
Path(out_dir).mkdir(parents=True, exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
).to('cuda')
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(TEST_PROMPTS):
    imgs = pipe(
        prompt,
        negative_prompt='photorealistic, blurry, cluttered, complex background, 3D render',
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=7.5,
        num_inference_steps=30,
        height=512,
        width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f'{out_dir}/prompt{i:02d}_v{j}.png')

del pipe; torch.cuda.empty_cache()
print(f'SD 1.5 baseline: {len(TEST_PROMPTS)*2} images -> {out_dir}')

In [ ]:
from diffusers import FluxPipeline
import torch
from pathlib import Path

flux_lora_path = f'{DRIVE_ROOT}/results/experiments/flux_r16'
out_dir = f'{DRIVE_ROOT}/results/experiments/flux_r16_samples'
Path(out_dir).mkdir(parents=True, exist_ok=True)

FLUX_PROMPTS = ['LOGOIMG ' + p.replace(', white background', '') for p in TEST_PROMPTS]

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-dev',
    torch_dtype=torch.bfloat16,
).to('cuda')
pipe.load_lora_weights(flux_lora_path)
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(FLUX_PROMPTS):
    imgs = pipe(
        prompt,
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=3.5,
        num_inference_steps=20,
        height=512, width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f'{out_dir}/prompt{i:02d}_v{j}.png')

del pipe; torch.cuda.empty_cache()
print(f'FLUX LoRA: {len(FLUX_PROMPTS)*2} images -> {out_dir}')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

BEST_SDXL_RANK = 16

MODELS = {
    'SD 1.5 Baseline': f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    f'SDXL LoRA r={BEST_SDXL_RANK}': f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_SDXL_RANK}_samples',
    'FLUX.1-dev LoRA': f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
}

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
for row in range(5):
    for col, (model_name, img_dir) in enumerate(MODELS.items()):
        img_path = f'{img_dir}/prompt{row:02d}_v0.png'
        if Path(img_path).exists():
            axes[row, col].imshow(Image.open(img_path))
        else:
            axes[row, col].text(0.5, 0.5, 'Not yet\ngenerated',
                                ha='center', va='center', fontsize=8)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(model_name, fontsize=10, fontweight='bold')

plt.suptitle('Experiment 3: SD1.5 Baseline vs SDXL LoRA vs FLUX LoRA', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/exp3_model_comparison.png', dpi=150)
plt.show()